In [ ]:
import wormtrails as wts
import cv2
import numpy as np
import pandas as pd

In [ ]:
# Load the raw video scan
video_array = wts.read_video_file("./chemotaxis_control.mp4")

In [ ]:
# Preprocessing of the scan
wts.correct_vignetting(video_array, inplace=True)
wts.subtract_average(video_array, inplace=True)

In [ ]:
# Preview your visualization parameters
wts.show_time_encoding(video_array, window=1, scale_factor=30, offset=-30, light_background=True)

In [ ]:
# Create and save your visualization
# In this case, an early and late timepoint are shown overlaid in different colors
early = wts.create_time_encoded_frame(video_array, colormap=np.array([[255,0,0]]), window=20, start_time=60*10, scale_factor=30, offset=-30, light_background=True)
late = wts.create_time_encoded_frame(video_array, colormap=np.array([[0,0,255]]), window=20, start_time=60*30, scale_factor=30, offset=-30, light_background=True)
early_and_late = np.min(np.array([early, late]), axis=0)
wts.show_frame(early_and_late)
cv2.imwrite("./early_and_late.png", early_and_late)

In [ ]:
# In this example, one timepoint is shown with fading trails
t_10_minutes = wts.create_time_encoded_frame(video_array, colormap=wts.white_to_black, window=20, start_time=60*10, scale_factor=30, offset=-30, light_background=True)
wts.show_frame(t_10_minutes)
cv2.imwrite("./t_10_minutes.png", t_10_minutes)

In [ ]:
# This example creates a video instead of a single frame
# Processing may take up to 15 minutes
time_encoded_array = wts.create_time_encoded_array(video_array[0:100], colormap=wts.white_to_black, window=20, scale_factor=30, offset=-30)

In [ ]:
# Add timestamp to video
wts.add_timestamp(time_encoded_array, black_background=False, font_scale=3, font_thickness=3, seconds_per_frame=1, inplace=True)

# Preview the created video
wts.show_video_array(time_encoded_array)

# Save the video
wts.write_mp4(time_encoded_array, "fading_trails.mp4", fps=30)

In [ ]:
# Quantitative measurements
# For chemotaxis, the positions, speeds, and direction of motion for each worm can be calculated at intervals over the scan
wts.threshold_array(video_array, 4, inplace=True) # The video array must have undergone average frame subtraction before this step
worm_data = wts.measure_chemotaxis(video_array[0:1200,:,:], time_window=10, interval=60, minimum_size=10, maximum_size=1000)
worm_data.to_csv("./worm_data.csv")

In [ ]:
# For lifespans, the number of detectably living worms on a plate can be calculated
# This step should not use an average subtracted video, since nearly dead worms will likely move very little
# A short video is recommended to avoid problems with overlaps, ideally with most worms moving less than 3 body lengths
video_array = wts.read_video_file("./chemotaxis_control.mp4")

n_alive, worms = wts.count_video(video_array[1140:1200], min_size=10, max_size=300, corrected_thresh=100, motion_thresh=3, kernel_size=11, detailed_output=True, plate_edge_size=None, plate_width=None)
print(n_alive)
wts.show_video_array(worms)